# Eddy pro to fluxy 

Convert the Eddy Pro text file output to a fluxy netcdf file.

Shows an example of processing and of variable names to use in the netcdf file.

In [ ]:
import pandas as pd
from pathlib import Path
from fluxy.test_utils import data_dir

In [ ]:
from collections import namedtuple
import numpy as np

eddy_txt_file = data_dir.parent / 'my_data' / "eddy_pro" / "Eddypro_V2.csv"
df = pd.read_csv(eddy_txt_file)

var_template = namedtuple(
    'var_template',
    [
        'name_in_file',
        'name_in_output',
        'unit',
    ]
)

sub = 'co2'

variables = [
    var_template(
        name_in_file='timeseries',
        name_in_output='time',
        unit='-'
    ),
    var_template(
        name_in_file=f"{sub}_flux",
        name_in_output="flux_observed",
        unit="umol m-2 s-1",
    ),
    var_template(
        name_in_file=f"qc_{sub}_flux",
        name_in_output="flux_observed_quality",
        unit="-",
    ),
    var_template(
        name_in_file=f"rand_err_{sub}_flux",
        name_in_output="flux_observed_random_error",
        unit="umol m-2 s-1",
    ),
    storage_var:= var_template(
        name_in_file=f"{sub}_strg2",
        name_in_output="flux_observed_storage",
        unit="umol m-2 s-1",

    ),
    var_template(
        name_in_file=f"{sub}_molar_density",
        name_in_output="md_observed",
        unit="mol m-3",
    ),
    var_template(
        name_in_file=f"{sub}_mole_fraction",
        name_in_output="mf_observed",
        unit="umol mol-1",
    ),
    var_template(
        name_in_file=f"{sub}_mixing_ratio",
        name_in_output="mr_observed",
        unit="umol mol-1",
    ),
    var_template(
        name_in_file="wind_speed",
        name_in_output="wind_speed",
        unit="m s-1",
    ),
    var_template(
        name_in_file="wind_dir",
        name_in_output="wind_direction",
        unit="degrees",
    ),
    var_template(
        name_in_file='air_temperature',
        name_in_output='air_temperature',
        unit='K',
    ),
    var_template(
        name_in_file='air_pressure',
        name_in_output='air_pressure',
        unit='Pa',
    ),
    var_template(
        name_in_file='pitch',
        name_in_output='pitch',
        unit='degrees',
    ),

]

df_this = df[[v.name_in_file for v in variables]].rename(
    columns={v.name_in_file: v.name_in_output for v in variables}
)
df_this['time'] = pd.to_datetime(df_this['time'], format='%Y/%m/%d %H:%M')
ds = df_this.to_xarray()

# Platform format (there is an index axis which is the main and then platforms (here just one))
# that are refered to with the number_of_identifier variable
ds = ds.assign(
    platform=('platform', ['Hardau']),
    number_of_identifier=('index', np.zeros(len(df_this), dtype=int)),
).set_coords(['time', 'number_of_identifier'])
# Assign units
for v in variables:
    if v.name_in_output == 'time':
        continue
    ds[v.name_in_output].attrs['units'] = v.unit

In [ ]:
import datetime
ds = ds.assign_attrs(
    {
        'title': 'Eddy covariance fluxes from Hardau',
        'institution': 'Empa/University of Basel',
        'source': 'Eddypro V2',
        'comment': 'Data processed with Eddypro V2 for fluxy plots',
        'creation_date': datetime.datetime.now().isoformat(),
        "fluxy_substance": sub,
        "fluxy_data_type": "eddy_flux",
        "fluxy_model": "EDDY_HARDAU",
    }
)
upper_sub = sub.upper()
out_file_path = eddy_txt_file.parent / 'EDDY' / upper_sub / f'EDDY_HARDAU_{upper_sub}_eddy_flux.nc'
out_file_path.parent.mkdir(parents=True, exist_ok=True)
ds.to_netcdf(out_file_path)

## Part 2: An additional storage file which only contains the storage flux variable

In [ ]:
# Another file which contains only the storage term
storage_file = eddy_txt_file.parent / "co2_strg_2layers_5point_20220801-20240731.csv"
df_storage = pd.read_csv(storage_file)
df_storage = df_storage.rename(
    columns={
        'Datetime_UTC_center': 'time',
        'co2_strg_2layers': storage_var.name_in_output,
    }
)
df_storage['time'] = pd.to_datetime(df_storage['time'], format='%Y-%m-%d %H:%M:%S')
ds_storage = df_storage.to_xarray().assign(
    platform=('platform', ['Hardau']),
    number_of_identifier=('index', np.zeros(len(df_storage), dtype=int)),
).set_coords(['time', 'number_of_identifier'])
# units 
ds_storage[storage_var.name_in_output].attrs['units'] = storage_var.unit
ds_storage = ds_storage.assign_attrs(
    {
        'title': 'Eddy covariance storage fluxes from Hardau with 2-layer model',
        'institution': 'Empa/University of Basel',
        'source': 'Eddypro V2',
        'comment': 'Data processed with Eddypro V2 for fluxy plots',
        'creation_date': datetime.datetime.now().isoformat(),
        "fluxy_substance": sub,
        "fluxy_data_type": "eddy_flux",
        "fluxy_model": "EDDY_HARDAU_STORAGE_2LAYERS",
    }
)
out_file_path_storage = eddy_txt_file.parent / 'EDDY' / upper_sub / f'EDDY_HARDAU_STORAGE_2LAYERS_{upper_sub}_eddy_flux.nc'
ds_storage.to_netcdf(out_file_path_storage)